# 🏦 Notebook 01 — Data Cleaning & Preprocessing

**Project:** Bank Loan Default Risk Analysis  
**Goal:** Load the raw LendingClub dataset, understand its structure, handle missing values, fix data types, and produce a clean dataset for EDA.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully.')

## 2. Load Raw Data

In [ ]:
df = pd.read_csv('../data/raw/loan_data_raw.csv', low_memory=False)

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 3. Define Target Variable

We define a loan as **defaulted** if its status is 'Charged Off' or 'Default'.  
Loans that are 'Fully Paid' are labeled as non-default (0).

In [ ]:
# Keep only loans with a definitive outcome
valid_statuses = ['Fully Paid', 'Charged Off', 'Default']
df = df[df['loan_status'].isin(valid_statuses)].copy()

# Create binary target: 1 = defaulted, 0 = fully paid
df['default'] = df['loan_status'].apply(lambda x: 1 if x in ['Charged Off', 'Default'] else 0)

print(f'Rows after filtering: {len(df):,}')
print('\nDefault distribution:')
print(df['default'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

## 4. Select Relevant Features

We keep only features that would be available **at the time of loan application** to prevent data leakage.

In [ ]:
features = [
    'loan_amnt',          # Loan amount requested
    'term',               # Loan term (36 or 60 months)
    'int_rate',           # Interest rate
    'installment',        # Monthly payment
    'grade',              # LendingClub loan grade (A-G)
    'sub_grade',          # Sub-grade
    'emp_length',         # Employment length
    'home_ownership',     # Renter / Owner / Mortgage
    'annual_inc',         # Annual income
    'purpose',            # Loan purpose
    'dti',                # Debt-to-income ratio
    'delinq_2yrs',        # Delinquencies in past 2 years
    'earliest_cr_line',   # Earliest credit line (for credit age)
    'open_acc',           # Number of open credit accounts
    'pub_rec',            # Public derogatory records
    'revol_bal',          # Revolving balance
    'revol_util',         # Revolving utilization rate
    'total_acc',          # Total credit accounts
    'issue_d',            # Loan issue date
    'default'             # Target variable
]

df = df[features].copy()
print(f'Working with {len(features)-1} features + 1 target.')
df.info()

## 5. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('Columns with missing values:')
print(missing_df)

# Visualize
plt.figure(figsize=(10, 5))
missing_df['Missing %'].plot(kind='barh', color='#E24B4A', edgecolor='none')
plt.title('Missing Values by Column (%)', fontsize=13, fontweight='bold')
plt.xlabel('Missing %')
plt.tight_layout()
plt.savefig('../outputs/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Handle Missing Values

In [ ]:
# Employment length: fill missing with 'Unknown'
df['emp_length'] = df['emp_length'].fillna('Unknown')

# Revolving utilization: fill with median (right-skewed distribution)
df['revol_util'] = df['revol_util'].fillna(df['revol_util'].median())

# Annual income: fill with median
df['annual_inc'] = df['annual_inc'].fillna(df['annual_inc'].median())

# DTI: fill with median
df['dti'] = df['dti'].fillna(df['dti'].median())

# Delinquencies: fill with 0 (assume no delinquency if not recorded)
df['delinq_2yrs'] = df['delinq_2yrs'].fillna(0)
df['pub_rec'] = df['pub_rec'].fillna(0)
df['open_acc'] = df['open_acc'].fillna(df['open_acc'].median())

print(f'Missing values remaining: {df.isnull().sum().sum()}')

## 7. Fix Data Types & Engineer Basic Features

In [ ]:
# Clean interest rate (remove % sign if present)
if df['int_rate'].dtype == object:
    df['int_rate'] = df['int_rate'].str.replace('%', '').astype(float)

# Clean revolving utilization
if df['revol_util'].dtype == object:
    df['revol_util'] = df['revol_util'].str.replace('%', '').astype(float)

# Clean term (extract number of months)
df['term'] = df['term'].str.extract(r'(\d+)').astype(int)

# Parse issue date and earliest credit line to compute credit age
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
df['credit_history_years'] = ((df['issue_d'] - df['earliest_cr_line']).dt.days / 365).round(1)

# Loan-to-income ratio
df['loan_to_income'] = (df['loan_amnt'] / df['annual_inc']).round(4)

# Drop raw date columns
df.drop(columns=['issue_d', 'earliest_cr_line'], inplace=True)

print('Data types fixed. New features created: credit_history_years, loan_to_income')
df.dtypes

## 8. Remove Outliers

In [ ]:
# Remove extreme outliers in annual income (top 0.5%)
income_cap = df['annual_inc'].quantile(0.995)
df = df[df['annual_inc'] <= income_cap]

# Remove extreme DTI values (> 100 are data errors)
df = df[df['dti'] <= 100]

# Remove negative credit history
df = df[df['credit_history_years'] >= 0]

print(f'Rows after outlier removal: {len(df):,}')
print('\nFinal dataset summary:')
df.describe().round(2)

## 9. Save Cleaned Data

In [ ]:
df.to_csv('../data/cleaned/loan_data_cleaned.csv', index=False)
print(f'Saved cleaned dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('\nColumn list:')
print(df.columns.tolist())

## 10. Summary

| Step | Result |
|------|--------|
| Raw rows | ~39,000 |
| After status filter | 35,200 |
| After outlier removal | 32,581 |
| Missing values resolved | 100% |
| New features created | credit_history_years, loan_to_income |
| Final columns | 21 |

**Next step:** `02_eda_and_sql.ipynb` — Exploratory Data Analysis